# Part 3: Connect OpenShell to Nemotron

In Part 2 Hermes Agent ran commands directly on your system. In this lab you use [OpenShell](https://build.nvidia.com/spark/openshell), which runs agents in isolated sandboxes. You create a sandbox that can reach your model server, send it a chat request, and see a request to another host denied.

This lab follows the NVIDIA playbook [OpenShell](https://build.nvidia.com/spark/openshell) with two differences. OpenShell 0.1.4 removed the `openshell inference` commands, and a provider type now needs an imported [provider profile](https://docs.nvidia.com/openshell/latest/how-it-works/providers/profiles).

This lab needs the model server from Part 2 to be running.


## 1. Check the model server

In [ ]:
import requests

model = requests.get("http://localhost:8000/v1/models", timeout=3).json()["data"][0]
print(f"Model: {model['id']}, context window: {model['max_model_len']:,} tokens")


## 2. Install OpenShell

The installer adds the `openshell` command and starts the OpenShell gateway as a background service. `openshell status` should report **Connected**.


In [ ]:
%%bash
curl -LsSf https://raw.githubusercontent.com/NVIDIA/OpenShell/main/install.sh | OPENSHELL_VERSION=v0.1.4 sh 2>&1
openshell status

## 3. Tell OpenShell about your model server

A provider profile declares which host and port a provider may reach. From the OpenShell docs: "A credential is only sent to the endpoints its profile declares." This cell writes a profile for your vLLM server, checks it with `lint`, and imports it.


In [ ]:
%%bash
IP=$(hostname -I | awk '{print $1}')
cat > ~/local-vllm-profile.yaml <<EOF
id: local-vllm
display_name: Local vLLM
description: Nemotron served by vLLM on this machine
category: inference
inference_capable: true
credentials:
  - name: api_key
    description: Placeholder key (vLLM does not check it)
    env_vars: [OPENAI_API_KEY]
    required: true
    auth_style: bearer
    header_name: authorization
discovery:
  credentials: [api_key]
endpoints:
  - host: $IP
    port: 8000
    protocol: rest
    access: read-write
    enforcement: enforce
binaries: [/usr/bin/curl, /usr/local/bin/curl]
EOF

openshell sandbox delete demo >/dev/null 2>&1
openshell provider delete local-vllm >/dev/null 2>&1
openshell profile delete local-vllm --global >/dev/null 2>&1

openshell profile lint -f ~/local-vllm-profile.yaml
openshell profile import -f ~/local-vllm-profile.yaml --global
openshell profile list


## 4. Create the provider

In [ ]:
%%bash
openshell provider create --name local-vllm --type local-vllm --credential OPENAI_API_KEY=not-needed
openshell provider list


## 5. List the models from inside a sandbox

The default sandbox image is minimal Ubuntu with no agent installed, and it has no `curl`. `--from` uses a different image, here `buildpack-deps:noble-curl`. The command after `--` runs inside the sandbox. The first run pulls the image.


In [ ]:
%%bash
IP=$(hostname -I | awk '{print $1}')
openshell sandbox delete demo >/dev/null 2>&1
openshell sandbox create --name demo --from buildpack-deps:noble-curl --provider local-vllm -- \
  curl -sS http://$IP:8000/v1/models


## 6. Ask the model a question from the sandbox

This cell sends a question to your model through the sandbox and prints the answer. It also counts the chat requests vLLM has logged before and after. If the count goes up by one, the request came from the sandbox.


In [ ]:
import json, subprocess

def vllm_requests():
    logs = subprocess.run(["docker", "logs", "vllm-nemotron-nano"], capture_output=True, text=True)
    return (logs.stdout + logs.stderr).count("POST /v1/chat/completions")

ip = subprocess.run("hostname -I | awk '{print $1}'", shell=True, capture_output=True, text=True).stdout.strip()
question = "What is the capital of France? Answer with one word."
body = json.dumps({"model": "nemotron", "messages": [{"role": "user", "content": question}], "max_tokens": 1000})

before = vllm_requests()
subprocess.run(["openshell", "sandbox", "delete", "demo"], capture_output=True)
run = subprocess.run(
    ["openshell", "sandbox", "create", "--name", "demo", "--from", "buildpack-deps:noble-curl",
     "--provider", "local-vllm", "--", "curl", "-sS", f"http://{ip}:8000/v1/chat/completions",
     "-H", "Content-Type: application/json", "-d", body],
    capture_output=True, text=True)
after = vllm_requests()

lines = (run.stdout + run.stderr).splitlines()
reply = next((l for l in lines if l.startswith("{")), None)
if reply is None:
    print("No reply from the model. OpenShell printed:\n" + "\n".join(lines))
else:
    answer = json.loads(reply)["choices"][0]["message"]["content"].strip()
    print(f"Question: {question}")
    print(f"Answer from nemotron, asked from inside the sandbox: {answer}")
    print(f"vLLM requests before: {before}, after: {after}")


## 7. Try a host the profile does not allow

The profile only allows your model server. This cell runs the same `curl` against `example.com` from a sandbox. The request should fail, and the OpenShell log should record why.


In [ ]:
import re, subprocess, time

subprocess.run(["openshell", "sandbox", "delete", "demo"], capture_output=True)
run = subprocess.run(
    ["openshell", "sandbox", "create", "--name", "demo", "--from", "buildpack-deps:noble-curl",
     "--provider", "local-vllm", "--", "curl", "-sS", "-m", "15", "https://example.com"],
    capture_output=True, text=True)

curl_line = next((l for l in (run.stdout + run.stderr).splitlines() if l.startswith("curl:")), "no curl error")
print(f"Request to https://example.com from the sandbox:\n  {curl_line}\n")

denied = policy = None
for _ in range(5):  # the log can lag the sandbox by a few seconds
    out = subprocess.run(["openshell", "logs", "demo"], capture_output=True, text=True)
    logs = re.sub(r"\x1b\[[0-9;]*m", "", out.stdout + out.stderr)
    denied = re.search(r"DENIED (\S+?)\(\d+\) -> (\S+) \[reason:(\S+)\]", logs)
    policy = re.search(r"endpoint \S+ is not allowed by any policy", logs)
    if denied:
        break
    time.sleep(2)

if denied:
    print("OpenShell denied it:")
    print(f"  program: {denied.group(1)}")
    print(f"  target:  {denied.group(2)}")
    print(f"  reason:  {denied.group(3)}")
    if policy:
        print(f"  log:     {policy.group(0)}")
else:
    print("No denial found in the OpenShell log. Last lines of the log:")
    print("\n".join(logs.splitlines()[-15:]))


## 8. Clean up

This deletes the sandbox, the provider and the provider profile. Leave the model server and gateway running.


In [ ]:
%%bash
openshell sandbox delete demo
openshell provider delete local-vllm
openshell profile delete local-vllm --global
